In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from sklearn.metrics import classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint

from tqdm import tqdm
# from mamba_ssm.modules.mamba2 import Mamba2

In [ ]:
# cuda_version = torch.version.cuda.replace(".", "") # e.g., '121'
torch_version = torch.__version__.split("+")[0] # e.g., '2.1.0'

# print(cuda_version)
print(torch_version)

In [ ]:
class BBFDataset(Dataset):
    def __init__(self, X_giant, y_giant, sequence_lengths):
        """
        X_giant: Tensor or Numpy array of shape (Total_Tokens, n_features)
        y_giant: Tensor or Numpy array of shape (Total_Tokens, 1)
        sequence_lengths: List or 1D array containing the true lengths of
                          each continuous sequence (e.g., [5, 16378, ...])
        """
        self.X = torch.tensor(X_giant, dtype=torch.float32)
        self.y = torch.tensor(y_giant, dtype=torch.float32)

        self.seqlens = sequence_lengths
        # Pre-calculate a global lookup map of start and end indices
        self.cu_seqlens_registry = [0] + torch.cumsum(torch.tensor(sequence_lengths), dim=0).tolist()

    def __len__(self):
        # Total number of available continuous sequences
        return len(self.seqlens)

    def __getitem__(self, idx):
        # Look up the absolute index boundaries for this specific sequence
        start_ptr = self.cu_seqlens_registry[idx]
        end_ptr = self.cu_seqlens_registry[idx + 1]

        # Pull the true slice directly from the giant flat tensor
        X_seq = self.X[start_ptr:end_ptr, :]
        y_seq_time = self.y[start_ptr:end_ptr]

        # Calculate sequence-level ground truth target dynamically over its true length
        y_seq_mean = y_seq_time.mean(dim=0)

        return X_seq, y_seq_time, y_seq_mean

In [ ]:
def giant_packed_collate_fn(batch):
    """
    Takes a batch of variable-length sequence slices and packs them back down
    into a single flat 1D sequence for efficient GPU execution.
    """
    X_list, y_time_list, y_seq_list = zip(*batch)

    # Calculate lengths of the sequences picked for this mini-batch
    seqlens = [len(x) for x in X_list]

    # Create the mini cu_seqlens boundary map for this specific batch
    cu_seqlens = torch.tensor([0] + torch.cumsum(torch.tensor(seqlens), dim=0).tolist(), dtype=torch.int32)

    # Concatenate along time dimension. Shape: (1, Total_Tokens_In_Batch, Features)
    X_packed = torch.cat(X_list, dim=0).unsqueeze(0)
    y_time_packed = torch.cat(y_time_list, dim=0)      # (Total_Tokens_In_Batch, 1)
    y_seq_stacked = torch.stack(y_seq_list, dim=0)     # (Batch_Size, 1)

    return X_packed, y_time_packed, y_seq_stacked, cu_seqlens

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
    
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        variance = x.pow(2).mean(-1, keepdim=True)
        
        return x * torch.rsqrt(variance + self.eps) * self.weight

In [ ]:
class BiMambaBlock(nn.Module):
    def __init__(self, d_model=128, d_state=32):
        super().__init__()

        self.fwd = Mamba2(
            d_model=d_model,
            d_state=d_state,
            d_conv=4,
            expand=2,
            headdim=64,
            ngroups=d_model // 64,
            chunk_size=256
        )

        self.bwd = Mamba2(
            d_model=d_model,
            d_state=d_state,
            d_conv=4,
            expand=2,
            headdim=64,
            ngroups=d_model // 64,
            chunk_size=256
        )

        self.merge = nn.Linear(d_model * 2, d_model)
        self.norm = RMSNorm(d_model)

    def forward(self, x, cu_seqlens):
        # x shape: (1, total_tokens, d_model)

        # forward pass
        x_fwd = self.fwd(x, cu_seqlens=cu_seqlens)

        # backward pass — reverse each sequence independently
        x_flat = x.squeeze(0)  # (total_tokens, d_model)
        batch_size = len(cu_seqlens) - 1
        reversed_seqs = []

        for i in range(batch_size):
            start = cu_seqlens[i]
            end = cu_seqlens[i + 1]
            reversed_seqs.append(torch.flip(x_flat[start:end], dims=[0]))

        x_rev_packed = torch.cat(reversed_seqs, dim=0).unsqueeze(0)
        x_bwd_raw = self.bwd(x_rev_packed, cu_seqlens=cu_seqlens).squeeze(0)

        # un-flip back to chronological order
        restored_seqs = []
        for i in range(batch_size):
            start = cu_seqlens[i]
            end = cu_seqlens[i + 1]
            restored_seqs.append(torch.flip(x_bwd_raw[start:end], dims=[0]))

        x_bwd = torch.cat(restored_seqs, dim=0).unsqueeze(0)

        # learned merge + pre-norm residual
        out = self.merge(torch.cat([x_fwd, x_bwd], dim=-1))
        return x + self.norm(out)

In [ ]:
class BEOLLoss(nn.Module):
    def __init__(self,
                 gamma: float = 0.05,
                 delta_max: float = 0.5,
                 beta: float = 3.0,
                 eta: float = 2.0,
                 delta: float = 0.01,
                 smooth: float = 1e-7):
        super().__init__()
        self.gamma = gamma
        self.delta_max = delta_max
        self.beta = beta
        self.eta = eta
        self.delta = delta
        self.smooth = smooth

    def _compute_p_star(self, p):
        # p shape: (1, T)
        T = p.size(1)
        if T <= 2:
            return torch.clamp(p, min=self.smooth, max=1.0 - self.smooth)

        interior = torch.abs(p[:, 2:] - 2.0 * p[:, 1:-1] + p[:, :-2])
        interior = torch.clamp(interior, max=self.delta_max)

        pad = torch.zeros((p.size(0), 1), dtype=torch.float32, device=p.device)
        delta2 = torch.cat([pad, interior, pad], dim=1)

        return torch.clamp(
            p - self.gamma * delta2,
            min=self.smooth,
            max=1.0 - self.smooth
        )

    def _compute_capture_rate(self, y_true, p):
        # Explicitly sum over the true sequence length dimension
        numerator = torch.sum(p * y_true, dim=1)
        denominator = torch.sum(y_true, dim=1) + self.smooth
        return numerator / denominator

    def _compute_occupancy(self, p):
        T = p.size(1)
        device = p.device

        # This will now safely be the individual sequence T (e.g., max 16,378)
        t_indices = torch.arange(1, T + 1, dtype=torch.float32, device=device).unsqueeze(0) # (1, T)

        cumsum_f = torch.cumsum(p, dim=1)
        F_t = cumsum_f / (t_indices + self.smooth)

        p_rev = torch.flip(p, dims=[1])
        cumsum_b = torch.cumsum(p_rev, dim=1)
        cumsum_b = torch.flip(cumsum_b, dims=[1])

        t_rev = torch.arange(T, 0, -1, dtype=torch.float32, device=device).unsqueeze(0) # (1, T)
        B_t = cumsum_b / (t_rev + self.smooth)

        D_t = torch.sqrt(F_t * B_t + self.delta)
        return F_t, B_t, D_t

    def _compute_psi(self, C, D_t):
        # C is (1,), D_t is (1, T)
        global_factor = (2.0 - C.unsqueeze(1)) ** 2
        local_factor = (1.0 - D_t) ** self.eta
        return 1.0 + global_factor * local_factor

    def _compute_phi(self, p, y_true, D_t):
        T = float(p.size(1))
        M_FP = torch.sum(p * (1.0 - y_true), dim=1) / T
        phi = torch.exp(
            self.beta * torch.log1p(M_FP.unsqueeze(1)) * (1.0 - D_t)
        )
        return phi

    def forward(self, y_pred, y_true, cu_seqlens):
        # 1. Ensure inputs are completely flat streams
        y_pred = y_pred.view(-1)
        y_true = y_true.view(-1)

        batch_size = len(cu_seqlens) - 1
        sequence_losses = []

        for i in range(batch_size):
            start = cu_seqlens[i]
            end = cu_seqlens[i+1]

            if start == end:
                continue

            # 2. CRITICAL FIX: Use .clone() so PyTorch isolates this slice's
            #    graph from the other 63 giant sequences in the batch stream.
            p = y_pred[start:end].clone().unsqueeze(0)
            y = y_true[start:end].clone().unsqueeze(0)

            p = torch.clamp(p, min=self.smooth, max=1.0 - self.smooth)

            # 3. Process metrics strictly for this independent sequence container
            p_star = self._compute_p_star(p)
            C = self._compute_capture_rate(y, p)
            _, _, D_t = self._compute_occupancy(p)

            psi = self._compute_psi(C, D_t)
            phi = self._compute_phi(p, y, D_t)

            event_loss = y * psi * torch.log(p_star + self.smooth)
            non_event_loss = (1.0 - y) * phi * torch.log(1.0 - p_star + self.smooth)

            # Mean loss over this sequence's isolated timesteps
            seq_loss = -torch.mean(event_loss + non_event_loss, dim=1)
            sequence_losses.append(seq_loss)

        if len(sequence_losses) == 0:
            return torch.tensor(0.0, device=y_pred.device, requires_grad=True)

        return torch.mean(torch.cat(sequence_losses))

In [ ]:
class PairwiseRankingLoss(nn.Module):
    def __init__(self, sigma: float = 1.0, weight_by_diff: bool = True):
        super().__init__()
        self.sigma = sigma
        self.weight_by_diff = weight_by_diff

    def forward(self, y_pred, y_true):
        """
        y_pred: (batch_size, 1) sequence predictions from the model
        y_true: (batch_size, 1) sequence mean ground truths from packed_collate_fn
        """
        # Ensure flat shape (Batch_Size,)
        y_true = y_true.view(-1)
        y_pred = y_pred.view(-1)

        # Pairwise differences matrix generation: (B, B)
        pred_diff = y_pred.unsqueeze(1) - y_pred.unsqueeze(0)
        true_diff = y_true.unsqueeze(1) - y_true.unsqueeze(0)

        # Target matrix extraction
        target_pij = (true_diff > 0).float()

        # Mask calculation separating diagonal frames and value equivalence ties
        mask = (torch.abs(true_diff) > 1e-7).float()
        mask = mask * (1.0 - torch.eye(y_true.size(0), device=y_true.device))

        # Sigmoid probability scaling using temperature hyperparameter
        pred_pij = torch.sigmoid(self.sigma * pred_diff)

        # Binary Cross Entropy over pairwise metrics
        pair_loss = (
            -target_pij * torch.log(pred_pij + 1e-7)
            - (1.0 - target_pij) * torch.log(1.0 - pred_pij + 1e-7)
        )

        # Scale cost landscape dynamically using raw parameter distances
        if self.weight_by_diff:
            pair_loss = pair_loss * torch.abs(true_diff)

        pair_loss = pair_loss * mask

        # Compute mean cost scaling cleanly over active pairs
        per_sample_loss = torch.sum(pair_loss, dim=1) / (torch.sum(mask, dim=1) + 1e-7)

        return torch.mean(per_sample_loss)

In [ ]:
class BBFModel(nn.Module):
    def __init__(self, n_features=15, d_model=128, d_state=32):
        super().__init__()

        # ── Multi-Scale Conv Frontend ──
        # local: kernel=5 — always applied, safe for min seq_len=5
        self.conv_local = nn.Conv1d(
            in_channels=n_features,
            out_channels=d_model // 2,
            kernel_size=5,
            padding=2
        )
        # medium: kernel=21 — applied when seq_len > 5
        self.conv_medium = nn.Conv1d(
            in_channels=n_features,
            out_channels=d_model // 2,
            kernel_size=21,
            padding=10
        )
        # fallback: linear projection for sequences with len <= 5
        self.conv_fallback = nn.Linear(n_features, d_model // 2)

        self.conv_act = nn.GELU()
        self.norm1 = RMSNorm(d_model)

        # ── 4 BiMamba Blocks ──
        self.blocks = nn.ModuleList([
            BiMambaBlock(d_model, d_state) for _ in range(4)
        ])

        # ── FFN ──
        self.ffn1 = nn.Linear(d_model, d_model * 2)
        self.ffn1_act = nn.GELU()
        self.ffn2 = nn.Linear(d_model * 2, d_model)
        self.ffn2_act = nn.GELU()
        self.skip_proj = nn.Linear(d_model, d_model)

        # ── Post Concat Fusion ──
        self.post_concat_dense = nn.Linear(d_model * 2, 128)
        self.post_concat_act = nn.GELU()

        # ── Output Heads ──
        self.time_output = nn.Linear(128, 1)
        self.seq_output = nn.Linear(128, 1)

    def _conv_frontend(self, x_flat, cu_seqlens):
        """
        Dynamic multi-scale conv applied per sequence.

        kernel=5  applied always — safe for min seq_len=5
        kernel=21 applied when seq_len > 5
        linear fallback for sequences with len <= 5

        x_flat: (total_tokens, n_features)
        Returns: (1, total_tokens, d_model)
        """
        batch_size = len(cu_seqlens) - 1
        outputs = []

        for i in range(batch_size):
            start = cu_seqlens[i]
            end = cu_seqlens[i + 1]
            seq_len = end - start

            # (1, n_features, seq_len) for Conv1d
            seq = x_flat[start:end].unsqueeze(0).transpose(1, 2)

            # local conv — always applied
            local_out = self.conv_local(seq)  # (1, d_model//2, seq_len)

            # medium conv or fallback
            if seq_len > 5:
                medium_out = self.conv_medium(seq)  # (1, d_model//2, seq_len)
            else:
                # linear fallback — (seq_len, d_model//2) → (1, d_model//2, seq_len)
                medium_out = self.conv_fallback(
                    x_flat[start:end]
                ).unsqueeze(0).transpose(1, 2)

            # concat channels → (1, d_model, seq_len)
            combined = torch.cat([local_out, medium_out], dim=1)

            # transpose → (seq_len, d_model)
            outputs.append(combined.squeeze(0).transpose(0, 1))

        # (1, total_tokens, d_model)
        return torch.cat(outputs, dim=0).unsqueeze(0)

    def forward(self, x, cu_seqlens):
        # x: (1, total_tokens, n_features)
        x_flat = x.squeeze(0)  # (total_tokens, n_features)

        # ── Dynamic Conv Frontend ──
        x = self._conv_frontend(x_flat, cu_seqlens)
        x = self.conv_act(x)
        x = self.norm1(x)

        # ── BiMamba Stack with Gradient Checkpointing ──
        for block in self.blocks:
            x = checkpoint(block, x, cu_seqlens, use_reentrant=False)

        skip = x

        # ── FFN ──
        x = self.ffn1_act(self.ffn1(x))
        x = self.ffn2_act(self.ffn2(x))
        skip = self.skip_proj(skip)

        # ── Fusion ──
        x = torch.cat([x, skip], dim=-1)                      # (1, total_tokens, d_model*2)
        x = self.post_concat_act(self.post_concat_dense(x))   # (1, total_tokens, 128)

        x = x.squeeze(0)  # (total_tokens, 128)

        # ── Time Output Head ──
        time_out = torch.sigmoid(self.time_output(x))  # (total_tokens, 1)

        # ── Sequence Output Head — segmented mean pooling ──
        batch_size = len(cu_seqlens) - 1
        seq_out_list = []

        for i in range(batch_size):
            start = cu_seqlens[i]
            end = cu_seqlens[i + 1]
            seq_mean = x[start:end].mean(dim=0)
            seq_out_list.append(seq_mean)

        x_seq = torch.stack(seq_out_list, dim=0)  # (batch_size, 128)
        seq_out = self.seq_output(x_seq)           # (batch_size, 1)

        return time_out, seq_out

In [ ]:
df = pd.read_parquet('data/v80_rescaled/final_model_data_thad_rescaled.parquet')
print("Loaded THEMIS A and D for Training")

In [ ]:
features = ['Bx', 'By', 'Bz', 'Bx_lag_1', 'Bx_lag_2', 'By_lag_1',
            'By_lag_2', 'Bz_lag_1', 'Bz_lag_2', 'Bx_conditional_vol',
            'By_conditional_vol', 'Bz_conditional_vol', 'LIM_scale_2_log',
            'LIM_scale_4_log', 'LIM_scale_8_log', 'LIM_scale_16_log',
            'LIM_scale_32_log', 'LIM_scale_64_log']
target = 'Event_label_80'

In [ ]:
X = df[features].values

y = df[target].values

In [ ]:
del df

In [ ]:
with open('/content/drive/MyDrive/Data/sequence_lengths_train.pkl', 'rb') as f:
    sequence_lengths_train = pickle.load(f)

In [ ]:
n_features = X.shape[1]

In [ ]:
train_dataset = BBFDataset(
    X_giant=X,                  # Your raw global X features
    y_giant=y,                  # Your raw global y targets
    sequence_lengths=sequence_lengths_train
)

# We pass the custom collate function to pack variable sequences into a 1D batch stream
batch_size = 64  # Keep this reasonable (16-32) because sequences scale up to 16k tokens

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=giant_packed_collate_fn,  # The crucial piece that builds cu_seqlens per batch
    pin_memory=True,
    num_workers=2
)

In [ ]:
# distribution of y_seq values in your DataLoader
# are they spread out or clustered?
y_seq_values = []
for _, _, y_seq in train_loader:
    y_seq_values.extend(y_seq.numpy().flatten())

import numpy as np
print(f"Mean: {np.mean(y_seq_values):.3f}")
print(f"Std: {np.std(y_seq_values):.3f}")
print(f"Min: {np.min(y_seq_values):.3f}")
print(f"Max: {np.max(y_seq_values):.3f}")

In [ ]:
criterion_time = BEOLLoss(
    gamma=0.05,
    delta_max=0.5,
    beta=5.0,
    eta=2.0,
    delta=0.01,
    smooth=1e-7
)

criterion_seq = PairwiseRankingLoss(
    sigma=1.0,
    weight_by_diff=True
)

num_epochs = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BBFModel(n_features=n_features, d_model=256, d_state=64).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

In [ ]:
print(f"Starting training on device: {device}\n")

for epoch in range(1, num_epochs + 1):
    model.train()

    running_total_loss = 0.0
    running_time_loss = 0.0
    running_seq_loss = 0.0

    running_tp = 0.0
    running_fp = 0.0
    running_fn = 0.0
    running_correct = 0.0
    running_total_tokens = 0.0

    progress_bar = tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc=f"EPOCH {epoch:>{len(str(num_epochs))}}/{num_epochs}",
        leave=True
    )

    for step, (X_batch, y_time_batch, y_seq_batch, cu_seqlens) in progress_bar:
        X_batch = X_batch.to(device)
        y_time_batch = y_time_batch.to(device)
        y_seq_batch = y_seq_batch.to(device)
        cu_seqlens = cu_seqlens.to(device)

        optimizer.zero_grad()

        pred_time, pred_seq = model(X_batch, cu_seqlens)

        pred_time = pred_time.view(-1)
        y_time_batch = y_time_batch.view(-1)

        pred_seq = pred_seq.view(-1)
        y_seq_batch = y_seq_batch.view(-1)

        loss_time = criterion_time(pred_time, y_time_batch, cu_seqlens)
        loss_seq = criterion_seq(pred_seq, y_seq_batch)

        total_loss = loss_time + loss_seq

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_total_loss += total_loss.item()
        running_time_loss += loss_time.item()
        running_seq_loss += loss_seq.item()

        with torch.no_grad():
            preds_binary = (pred_time >= 0.5).float()
            targets_binary = y_time_batch.float()

            running_correct += (preds_binary == targets_binary).sum().item()
            running_total_tokens += targets_binary.numel()
            running_tp += (preds_binary * targets_binary).sum().item()
            running_fp += (preds_binary * (1.0 - targets_binary)).sum().item()
            running_fn += ((1.0 - preds_binary) * targets_binary).sum().item()

            current_acc = running_correct / (running_total_tokens + 1e-7)
            current_precision = running_tp / (running_tp + running_fp + 1e-7)
            current_recall = running_tp / (running_tp + running_fn + 1e-7)

        step_count = step + 1

        progress_bar.set_postfix({
            "loss": f"{running_total_loss / step_count:.4f}",
            "seq": f"{running_seq_loss / step_count:.4f}",
            "time": f"{running_time_loss / step_count:.4f}",
            "accuracy": f"{current_acc:.4f}",
            "precision": f"{current_precision:.4f}",
            "recall": f"{current_recall:.4f}",
        })

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': running_total_loss / len(train_loader),
        'precision': current_precision,
        'recall': current_recall,
    }, f'/content/drive/MyDrive/Data/mamba_checkpoints/model_{epoch}.pt')

    print()

In [ ]:
del X
del y

print("Deleted training data variables")

In [ ]:
df_the = pd.read_parquet('data/v80_rescaled/final_model_data_the_rescaled.parquet')
print("Loaded THEMIS E for Testing")

In [ ]:
# create X_the and y_the for evaluation
X = df_the[features].values
y = df_the[target].values

In [ ]:
del df_the

In [ ]:
with open('/content/drive/MyDrive/Data/sequence_lengths_test.pkl', 'rb') as f:
    sequence_lengths_test = pickle.load(f)

In [ ]:
test_dataset = BBFDataset(
    X_giant=X,                  # Your raw global X features
    y_giant=y,                  # Your raw global y targets
    sequence_lengths=sequence_lengths_test
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=giant_packed_collate_fn,
    pin_memory=True,
    num_workers=2
)

In [ ]:
model.eval()

time_preds_list = []
seq_preds_list = []

progress_bar = tqdm(test_loader, desc="PREDICTING")

with torch.no_grad():
    # 1. Unpack cu_seqlens instead of ignoring it
    for X_batch, _, _, cu_seqlens in progress_bar:
        X_batch = X_batch.to(device)
        # 2. Ensure cu_seqlens is on the same device as the model/data
        cu_seqlens = cu_seqlens.to(device)

        # 3. Pass cu_seqlens into the model's forward call
        pred_time, pred_seq = model(X_batch, cu_seqlens)

        # Move tensors back to CPU and convert to NumPy arrays immediately
        time_preds_list.append(pred_time.cpu().numpy())
        seq_preds_list.append(pred_seq.cpu().numpy())

# Concatenate all batches into single continuous NumPy arrays
y_pred_probas = np.concatenate(time_preds_list, axis=0)

In [ ]:
plt.hist(y_pred_probas[y == 0], bins=50, alpha=0.5, label='Negative Class', edgecolor='black')
plt.hist(y_pred_probas[y == 1], bins=50, alpha=0.5, label='Positive Class', edgecolor='black')
plt.xlabel('Predicted Probability')
plt.ylabel('Frequency')
plt.title('Histogram of Predicted Probabilities')
plt.legend()
plt.show()

In [ ]:
threshold = 0.5
y_pred = (y_pred_probas >= threshold).astype(int)
print(classification_report(y, y_pred))